In [5]:
from pathlib import Path

base_path = Path("/data/CARDPB2/iNDI/JaneliaTest/organelle_features")
base_files = list(base_path.glob("*.png"))[:5]

print(base_files)

[]


In [4]:
# move files that start with endosome into their own endosome folder - repeat for each organelle (class)
from pathlib import Path
import shutil

base_path = Path("/data/CARDPB2/iNDI/JaneliaTest/organelle_features")

for prefix in ["Endosome", "Lysosome", "Mitochondria"]:
    folder = base_path / prefix
    folder.mkdir(exist_ok=True)
    for f in base_path.glob(f"{prefix}*.png"):
        shutil.move(f, folder / f.name)

In [9]:
endosome_path = Path("/data/CARDPB2/iNDI/JaneliaTest/organelle_features/Endosome")
mitochondria_path = Path("/data/CARDPB2/iNDI/JaneliaTest/organelle_features/Mitochondria")
lysosome_path = Path("/data/CARDPB2/iNDI/JaneliaTest/organelle_features/Lysosome")

list(lysosome_path.glob("*.png"))[:5]

[PosixPath('/data/CARDPB2/iNDI/JaneliaTest/organelle_features/Lysosome/Lysosome__r01c03f01p01-ch01t01.tiff__NucID=12.png'),
 PosixPath('/data/CARDPB2/iNDI/JaneliaTest/organelle_features/Lysosome/Lysosome__r01c03f01p01-ch01t01.tiff__NucID=13.png'),
 PosixPath('/data/CARDPB2/iNDI/JaneliaTest/organelle_features/Lysosome/Lysosome__r01c03f01p01-ch01t01.tiff__NucID=18.png'),
 PosixPath('/data/CARDPB2/iNDI/JaneliaTest/organelle_features/Lysosome/Lysosome__r01c03f01p01-ch01t01.tiff__NucID=22.png'),
 PosixPath('/data/CARDPB2/iNDI/JaneliaTest/organelle_features/Lysosome/Lysosome__r01c03f01p01-ch01t01.tiff__NucID=26.png')]

In [1]:
from pathlib import Path
import shutil
import random
import re
from collections import defaultdict
from tqdm import tqdm

base_path = Path("/data/CARDPB2/iNDI/JaneliaTest/organelle_features")
dest_path = Path("/data/CARDPB2/iNDI/JaneliaTest/organelle_features_subset")

prefixes = ["Endosome_crop", "Lysosome_crop", "Mitochondria_crop"]

pattern = re.compile(r'__(r\d+c\d+)(f\d+p\d+-ch\d+t\d+)\.tiff__NucID=(\d+)_c\d+\.png$')

# Scan all folders, storing wells and full file paths
well_sets = []
all_files = {}  # prefix -> {well -> {frame -> [filepaths]}}

for prefix in prefixes:
    src_folder = base_path / prefix
    wells_found = set()
    file_map = defaultdict(lambda: defaultdict(list))

    files = list(src_folder.iterdir())
    for f in tqdm(files, desc=f"Scanning {prefix}", unit="file"):
        m = pattern.search(f.name)
        if m:
            well, frame = m.group(1), m.group(2)
            wells_found.add(well)
            file_map[well][frame].append(f)

    well_sets.append(wells_found)
    all_files[prefix] = file_map
    print(f"{prefix}: {len(wells_found)} wells found")

# Find common wells and select 10
common_wells = sorted(set.intersection(*well_sets))
print(f"\n{len(common_wells)} wells common to all organelles")

if len(common_wells) < 10:
    print(f"WARNING: only {len(common_wells)} common wells available, using all")
    wells = common_wells
else:
    wells = random.sample(common_wells, 10)

print(f"Selected wells: {wells}")

# For each well, find frames common to all organelles, then pick 5 random
selected = defaultdict(list)
for well in wells:
    frame_sets = [set(all_files[p][well].keys()) for p in prefixes]
    common_frames = set.intersection(*frame_sets)
    if len(common_frames) < 5:
        print(f"  WARNING: only {len(common_frames)} common frames for {well}, using all")
        selected[well] = list(common_frames)
    else:
        selected[well] = random.sample(sorted(common_frames), 5)

print(f"\nSelected frames per well:")
for well, frames in selected.items():
    print(f"  {well}: {frames}")

# Copy using the already-built file map — no glob needed
total_copied = 0
for prefix in prefixes:
    dst_folder = dest_path / prefix
    dst_folder.mkdir(parents=True, exist_ok=True)

    to_copy = []
    for well, frames in selected.items():
        for frame in frames:
            to_copy.extend(all_files[prefix][well][frame])

    for f in tqdm(to_copy, desc=f"Copying {prefix}", unit="file"):
        shutil.copy2(f, dst_folder / f.name)
        total_copied += 1

print(f"\nDone — {total_copied} files copied to {dest_path}")

Scanning Endosome_crop: 100%|██████████| 2445282/2445282 [00:02<00:00, 851456.23file/s]


Endosome_crop: 352 wells found


Scanning Lysosome_crop: 100%|██████████| 3295476/3295476 [00:04<00:00, 823687.05file/s]


Lysosome_crop: 288 wells found


Scanning Mitochondria_crop: 100%|██████████| 4088770/4088770 [00:04<00:00, 840840.60file/s]


Mitochondria_crop: 160 wells found

160 wells common to all organelles
Selected wells: ['r02c24', 'r07c12', 'r07c06', 'r04c03', 'r01c19', 'r04c19', 'r03c08', 'r01c21', 'r05c19', 'r01c15']

Selected frames per well:
  r02c24: ['f33p01-ch01t01', 'f13p01-ch01t01', 'f11p01-ch01t01', 'f35p01-ch01t01', 'f27p01-ch01t01']
  r07c12: ['f08p01-ch01t01', 'f10p01-ch01t01', 'f24p01-ch01t01', 'f19p01-ch01t01', 'f23p01-ch01t01']
  r07c06: ['f23p01-ch01t01', 'f02p01-ch01t01', 'f13p01-ch01t01', 'f11p01-ch01t01', 'f01p01-ch01t01']
  r04c03: ['f12p01-ch01t01', 'f27p01-ch01t01', 'f23p01-ch01t01', 'f17p01-ch01t01', 'f15p01-ch01t01']
  r01c19: ['f33p01-ch01t01', 'f05p01-ch01t01', 'f24p01-ch01t01', 'f07p01-ch01t01', 'f30p01-ch01t01']
  r04c19: ['f10p01-ch01t01', 'f29p01-ch01t01', 'f12p01-ch01t01', 'f02p01-ch01t01', 'f04p01-ch01t01']
  r03c08: ['f26p01-ch01t01', 'f02p01-ch01t01', 'f16p01-ch01t01', 'f35p01-ch01t01', 'f19p01-ch01t01']
  r01c21: ['f09p01-ch01t01', 'f34p01-ch01t01', 'f05p01-ch01t01', 'f01p01-ch01t

Copying Mitochondria_crop: 100%|██████████| 35986/35986 [04:29<00:00, 133.57file/s]


Done — 63104 files copied to /data/CARDPB2/iNDI/JaneliaTest/organelle_features_subset


In [1]:
from pathlib import Path
import shutil
import random
import re
from collections import defaultdict
from tqdm import tqdm
from PIL import Image

base_path = Path("/data/CARDPB2/iNDI/JaneliaTest/organelle_features")
dest_path = Path("/data/CARDPB2/iNDI/JaneliaTest/genotype_subset")

organelles = ["Endosome", "Lysosome", "Mitochondria"]

# 384-well plate: rows A-P = r01-r16, cols 01-24 = c01-c24
def well_to_rc(well):
    row_letter = well[0]
    col = int(well[1:])
    row = ord(row_letter) - ord('A') + 1
    return f"r{row:02d}c{col:02d}"

genotypes = {
    "Parental":           ["B07", "C12", "D17", "F02", "H07"],
    "TBK1_E696K_HET":    ["B14", "C13", "F16", "J13", "M04"],
    "TBK1_E696K_HOM":    ["B22", "E18", "H16", "K13", "N23"],
    "TBK1_E696K_REV":    ["B19", "G08", "K01", "O13", "P11"],
    "HNRNPA1_D262N_HOM": ["C18", "E09", "F11", "J15", "O10"],
    "HNRNPA1_D262N_REV": ["A16", "E21", "H15", "I20", "N03"],
}

# Convert well IDs to rXXcXX format
genotypes_rc = {g: [well_to_rc(w) for w in wells] for g, wells in genotypes.items()}

pattern_roi  = re.compile(r'__(r\d+c\d+)(f\d+p\d+-ch\d+t\d+)\.tiff__NucID=(\d+)\.png$')
pattern_crop = re.compile(r'__(r\d+c\d+)(f\d+p\d+-ch\d+t\d+)\.tiff__NucID=(\d+)_c\d+\.png$')

def check_size(f, size=(241, 241)):
    try:
        with Image.open(f) as img:
            return img.size == size
    except Exception:
        return False

# --- Step 1: Use Endosome ROIs to find valid 240x240 frames per genotype ---
print("Scanning Endosome ROIs for 241x241 frames...")
roi_folder = base_path / "Endosome"

# Build map: well -> frame -> [roi filepaths]
roi_map = defaultdict(lambda: defaultdict(list))
roi_files = list(roi_folder.iterdir())
for f in tqdm(roi_files, desc="Scanning Endosome ROIs", unit="file"):
    m = pattern_roi.search(f.name)
    if m:
        well, frame = m.group(1), m.group(2)
        roi_map[well][frame].append(f)

# For each genotype, find valid frames (241x241) across its wells, pick 5
selected = {}  # genotype -> [(well, frame), ...]

for genotype, wells in genotypes_rc.items():
    valid_frames = []
    for well in wells:
        for frame, files in roi_map[well].items():
            # Check first file for this well+frame
            if files and check_size(files[0]):
                valid_frames.append((well, frame))

    if len(valid_frames) < 5:
        print(f"  WARNING: {genotype} only has {len(valid_frames)} valid 241x241 frames, using all")
        selected[genotype] = valid_frames
    else:
        selected[genotype] = random.sample(valid_frames, 5)

    print(f"  {genotype}: {len(valid_frames)} valid frames, selected {len(selected[genotype])}")

print("\nSelected frames per genotype:")
for genotype, frames in selected.items():
    print(f"  {genotype}: {frames}")

# --- Step 2: Scan crop folders and copy matching files ---
for organelle in organelles:
    crop_folder = base_path / f"{organelle}_crop"
    print(f"\nScanning {organelle}_crop...")

    # Build map: well -> frame -> [crop filepaths]
    crop_map = defaultdict(lambda: defaultdict(list))
    crop_files = list(crop_folder.iterdir())
    for f in tqdm(crop_files, desc=f"Scanning {organelle}_crop", unit="file"):
        m = pattern_crop.search(f.name)
        if m:
            well, frame = m.group(1), m.group(2)
            crop_map[well][frame].append(f)

    # Copy crops for each genotype
    for genotype, frames in selected.items():
        dst_folder = dest_path / organelle / genotype
        dst_folder.mkdir(parents=True, exist_ok=True)

        to_copy = []
        for well, frame in frames:
            to_copy.extend(crop_map[well][frame])

        for f in tqdm(to_copy, desc=f"  Copying {genotype}", unit="file"):
            shutil.copy2(f, dst_folder / f.name)

print("\nDone!")

Scanning Endosome ROIs for 240x240 frames...


Scanning Endosome ROIs: 100%|██████████| 95365/95365 [00:00<00:00, 623289.50file/s]


  Parental: 91 valid frames, selected 5
  TBK1_E696K_HET: 105 valid frames, selected 5
  TBK1_E696K_HOM: 102 valid frames, selected 5
  TBK1_E696K_REV: 134 valid frames, selected 5
  HNRNPA1_D262N_HOM: 135 valid frames, selected 5
  HNRNPA1_D262N_REV: 83 valid frames, selected 5

Selected frames per genotype:
  Parental: [('r02c07', 'f24p01-ch01t01'), ('r08c07', 'f33p01-ch01t01'), ('r06c02', 'f29p01-ch01t01'), ('r02c07', 'f04p01-ch01t01'), ('r08c07', 'f35p01-ch01t01')]
  TBK1_E696K_HET: [('r06c16', 'f05p01-ch01t01'), ('r03c13', 'f02p01-ch01t01'), ('r03c13', 'f12p01-ch01t01'), ('r13c04', 'f11p01-ch01t01'), ('r06c16', 'f04p01-ch01t01')]
  TBK1_E696K_HOM: [('r02c22', 'f08p01-ch01t01'), ('r08c16', 'f05p01-ch01t01'), ('r05c18', 'f11p01-ch01t01'), ('r14c23', 'f26p01-ch01t01'), ('r11c13', 'f14p01-ch01t01')]
  TBK1_E696K_REV: [('r02c19', 'f22p01-ch01t01'), ('r16c11', 'f25p01-ch01t01'), ('r15c13', 'f33p01-ch01t01'), ('r07c08', 'f30p01-ch01t01'), ('r11c01', 'f10p01-ch01t01')]
  HNRNPA1_D262N_HOM

  Copying HNRNPA1_D262N_REV: 100%|██████████| 142/142 [00:00<00:00, 143.70file/s]



Scanning Lysosome_crop...


  Copying HNRNPA1_D262N_REV: 100%|██████████| 151/151 [00:01<00:00, 133.43file/s]



Scanning Mitochondria_crop...


  Copying HNRNPA1_D262N_REV: 100%|██████████| 329/329 [00:02<00:00, 139.57file/s]


Done!
